In [ ]:
!rm -rf Classical-Chinese-Summarization
!git clone https://github.com/ctaiyi15/Classical-Chinese-Summarization.git
!ls Classical-Chinese-Summarization/data

Cloning into 'Classical-Chinese-Summarization'...
remote: Enumerating objects: 2665, done.
remote: Counting objects: 100% (2665/2665), done.
remote: Compressing objects: 100% (1397/1397), done.
remote: Total 2665 (delta 48), reused 2634 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (2665/2665), 19.64 MiB | 20.50 MiB/s, done.
Resolving deltas: 100% (48/48), done.
processed  raw


In [ ]:
!ls Classical-Chinese-Summarization/data/processed

evaluation  sample  summary  translated  translated_back


In [ ]:
from openai import OpenAI
import json
import re
from google.colab import userdata
from tqdm.notebook import tqdm

client = OpenAI(
    api_key=userdata.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

def split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sentences if s.strip()]

def extract_claims(summary):
  prompt = f"""
You are a precise information extraction system.

Task:
Break the summary into atomic factual claims.

Rules:
- Each claim must contain ONE fact only
- No merging multiple facts
- Do NOT add new information
- Keep claims short and concrete

Return JSON ONLY:
{{
  "claims": ["claim 1", "claim 2", ...]
}}

SUMMARY:
{summary}
"""

  response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

  return json.loads(response.choices[0].message.content)["claims"]

def retrieve_relevant_parts(claim, source_text):
    prompt = f"""
You are an information retrieval system.

Task:
Extract ONLY the parts of the source text that are relevant to verifying the claim.

Rules:
- Copy exact spans from the source text
- Do NOT paraphrase
- If nothing is relevant, return empty list
- Keep output minimal (max 5 excerpts)

Return JSON ONLY:
{{
  "evidence": ["excerpt 1", "excerpt 2", ...]
}}

SOURCE TEXT:
{source_text}

CLAIM:
{claim}
"""

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return json.loads(response.choices[0].message.content)["evidence"]

def verify_claim(claim, evidence_list):
    context = "\n\n".join(evidence_list)

    prompt = f"""
You are a strict fact-checking system.

Task:
Determine whether the claim is supported by the evidence.

Rules:
- ONLY use the provided evidence
- If evidence is insufficient → mark as unsupported
- Paraphrases are allowed if meaning is identical
- Be strict: partial mismatch = unsupported

Return JSON ONLY:
{{
  "label": "supported / unsupported"
}}

EVIDENCE:
{context}

CLAIM:
{claim}
"""

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return json.loads(response.choices[0].message.content)

def evaluate_faithfulness(summary, source_text):
    claims = extract_claims(summary)

    results = []

    for claim in claims:
        evidence = retrieve_relevant_parts(claim, source_text)
        verdict = verify_claim(claim, evidence)

        results.append({
            "claim": claim,
            "evidence": evidence,
            "label": verdict["label"]
        })

    supported = sum(r["label"] == "supported" for r in results)
    total = len(results)

    score = supported / total if total > 0 else 0

    return {
        "faithfulness_score": score,
        "total_claims": total,
        "supported_claims": supported,
        "details": results
    }


def run_dataset(summary_source_pairs):
    outputs = []

    for summary, source in tqdm(summary_source_pairs):
        try:
            result = evaluate_faithfulness(summary, source)
            outputs.append(result)
        except Exception as e:
            outputs.append({"error": str(e)})

    return outputs

from pathlib import Path

input_root = Path("Classical-Chinese-Summarization/data/processed/translated")
summary_root = Path("Classical-Chinese-Summarization/data/processed/summary")

def build_pairs():
    pairs = []

    source_files = list(input_root.rglob("*.txt"))

    for src_path in source_files:
        rel = src_path.relative_to(input_root)
        summary_path = (summary_root / rel).with_suffix(".summary.txt")

        if summary_path.exists():
            pairs.append((src_path, summary_path))

    return pairs

def read_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

def evaluate_faithfulness(summary, source_text):
    claims = extract_claims(summary)

    results = []

    for claim in claims:
        evidence = retrieve_relevant_parts(claim, source_text)
        verdict = verify_claim(claim, evidence)

        results.append({
            "claim": claim,
            "evidence": evidence,
            "label": verdict["label"]
        })

    supported = sum(r["label"] == "supported" for r in results)
    total = len(results)

    score = supported / total if total > 0 else 0

    return {
        "faithfulness_score": score,
        "total_claims": total,
        "supported_claims": supported,
        "details": results
    }

import json
from tqdm.notebook import tqdm

def run_dataset(save_path="faithfulness_results.json"):
    pairs = build_pairs()

    results = []
    total = len(pairs)

    print(f"Found {total} file pairs")

    for src_path, sum_path in tqdm(pairs):
        try:
            source_text = read_file(src_path)
            summary_text = read_file(sum_path)

            result = evaluate_faithfulness(summary_text, source_text)

            results.append({
                "source_file": str(src_path),
                "summary_file": str(sum_path),
                **result
            })

        except Exception as e:
            results.append({
                "source_file": str(src_path),
                "summary_file": str(sum_path),
                "error": str(e)
            })

    # save full results
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\nSaved results to: {save_path}")

    return results

# results = run_dataset()

import json
from pathlib import Path

LIVE_SAVE_PATH = "faithfulness_live_results.json"

def run_first_20_live_saved():
    pairs = build_pairs()[:20]

    results = []

    print(f"Running evaluation on {len(pairs)} file pairs...\n")

    for i, (src_path, sum_path) in enumerate(pairs):

        print("=" * 60)
        print(f"[{i+1}/20] {src_path.name}")

        source_text = read_file(src_path)
        summary_text = read_file(sum_path)

        result = evaluate_faithfulness(summary_text, source_text)

        score = result["faithfulness_score"]

        entry = {
            "source_file": str(src_path),
            "summary_file": str(sum_path),
            **result
        }

        results.append(entry)

        # live print
        print("Score:", score)

        # running average
        avg = sum(r["faithfulness_score"] for r in results) / len(results)
        print("Running avg:", avg)

        with open(LIVE_SAVE_PATH, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

        print("Saved checkpoint →", LIVE_SAVE_PATH)

    print("\nDONE")
    print("Final avg:", sum(r["faithfulness_score"] for r in results) / len(results))

run_first_20_live_saved()



Running evaluation on 20 file pairs...

[1/20] target.txt
Score: 0.6710526315789473
Running avg: 0.6710526315789473
Saved checkpoint → faithfulness_live_results.json
[2/20] target.txt
Score: 0.7464788732394366
Running avg: 0.7087657524091919
Saved checkpoint → faithfulness_live_results.json
[3/20] target.txt
Score: 0.717391304347826
Running avg: 0.7116409363887367
Saved checkpoint → faithfulness_live_results.json
[4/20] target.txt
Score: 0.6115702479338843
Running avg: 0.6866232642750236
Saved checkpoint → faithfulness_live_results.json
[5/20] target.txt
Score: 0.6846846846846847
Running avg: 0.6862355483569558
Saved checkpoint → faithfulness_live_results.json
[6/20] target.txt
Score: 0.6666666666666666
Running avg: 0.6829740680752409
Saved checkpoint → faithfulness_live_results.json
[7/20] target.txt
Score: 0.5942028985507246
Running avg: 0.6702924724288815
Saved checkpoint → faithfulness_live_results.json
[8/20] target.txt
Score: 0.7924528301886793
Running avg: 0.6855625171488562
Sav